# Phase 3 : nettoyage et transformation

Ce notebook part des fichiers bruts et produit les tables nettoyées. Il applique dans l'ordre les fonctions de `src/clean_data.py`. Chaque étape est expliquée dans `docs/`.

Avant de commiter : **Clear All Outputs**, car certaines sorties peuvent afficher des données sensibles.

## Chargement des données (P3-01)

In [ ]:
import pandas as pd

from src.load_data import (
    load_transactions, load_cards, load_users, load_mcc_codes, load_fraud_labels,
)
from src.clean_data import (
    convert_currency_column, report_conversion_issues, convert_month_year_column,
    check_datetime_column, build_mcc_table, check_mcc_join, add_mcc_families,
    check_mcc_families, transform_errors_column, check_errors_transformation,
    structurer_lieu_transactions, check_location_structuration,
    flag_suspicious_transactions, flag_suspicious_users, flag_suspicious_cards,
    check_suspicious_flags,
)

pd.set_option("display.max_colwidth", None)

In [ ]:
df_transaction = load_transactions()
df_card = load_cards()
df_users = load_users()
df_mcc = load_mcc_codes()
df_labels = load_fraud_labels()

for name, df in [("transactions", df_transaction), ("cards", df_card), ("users", df_users),
                 ("mcc", df_mcc), ("fraud_labels", df_labels)]:
    print(f"{name}: {df.shape[0]:,} lignes, {df.shape[1]} colonnes")
print(f"Mémoire des transactions : {df_transaction.memory_usage(deep=True).sum() / 1e6:.1f} Mo")

**Constats**

- Les cinq tables sont chargées : 13 305 915 transactions, 6 146 cartes, 2 000 clients, 109 codes MCC et 8 914 963 étiquettes.
- La table des transactions occupe 1 371,0 Mo en mémoire.

## P3-02 : conversion des montants

Critère de fin : aucune valeur non convertie, ou cas listés. Voir `docs/02_conversion_montants.md`.

In [ ]:
# transactions_data
original = df_transaction["amount"].copy()
df_transaction["amount"] = convert_currency_column(df_transaction["amount"])
report_conversion_issues(original, df_transaction["amount"], "transactions.amount")

# cards_data
original = df_card["credit_limit"].copy()
df_card["credit_limit"] = convert_currency_column(df_card["credit_limit"])
report_conversion_issues(original, df_card["credit_limit"], "cards.credit_limit")

# users_data
for col in ["per_capita_income", "yearly_income", "total_debt"]:
    original = df_users[col].copy()
    df_users[col] = convert_currency_column(df_users[col])
    report_conversion_issues(original, df_users[col], f"users.{col}")
del original

**Constats**

- Les cinq colonnes de montants sont converties : 0 valeur non convertie dans chacune.

## P3-03 : conversion des dates

Critère de fin : aucune date non convertie. Voir `docs/03_conversion_dates.md`.

In [ ]:
original = df_card["expires"].copy()
df_card["expires"] = convert_month_year_column(df_card["expires"], end_of_month=True)
report_conversion_issues(original, df_card["expires"], "cards.expires")

original = df_card["acct_open_date"].copy()
df_card["acct_open_date"] = convert_month_year_column(df_card["acct_open_date"])
report_conversion_issues(original, df_card["acct_open_date"], "cards.acct_open_date")
del original

check_datetime_column(df_card["expires"], "cards.expires")
check_datetime_column(df_card["acct_open_date"], "cards.acct_open_date")
check_datetime_column(df_transaction["date"], "transactions.date")

**Constats**

- `expires` et `acct_open_date` sont converties : 0 valeur non convertie.
- Aucune date ne manque dans les trois colonnes de dates.
- Les transactions vont du 1er janvier 2010 au 31 octobre 2019. Les ouvertures de compte vont de janvier 1991 à février 2020. Les expirations vont de juillet 1997 à décembre 2024.

## P3-04 et P3-05 : table des catégories MCC et familles

Critères de fin : jointure avec les transactions sans perte ; table de correspondance validée. Voir `docs/04_table_mcc.md` et `docs/05_familles_mcc.md`.

In [ ]:
df_mcc = build_mcc_table(df_mcc)
ok_jointure = check_mcc_join(df_transaction, df_mcc)
print("Jointure sans perte :", ok_jointure)

df_mcc = add_mcc_families(df_mcc)
ok_familles = check_mcc_families(df_mcc, df_transaction)
print("Table de correspondance valide :", ok_familles)

**Constats**

- La jointure garde les 13 305 915 transactions, et aucune n'est sans catégorie.
- Les 109 codes ont une famille et une description en français.
- Les deux plus grandes familles sont « Transport, carburant et automobile » (3 352 312 transactions, 25,2 %) et « Alimentation » (3 182 334, 23,9 %). La plus petite est « Voyage et hébergement » (52 026, 0,4 %).

## P3-06 : transformation de la colonne errors

Critère de fin : totaux identiques à ceux de P2-13. Voir `docs/06_colonne_errors.md`.

In [ ]:
df_transaction = transform_errors_column(df_transaction)
check_errors_transformation(df_transaction)

**Constats**

- Les totaux des 7 colonnes d'erreur sont identiques à ceux de P2-13.
- 211 393 transactions ont au moins une erreur, dont 53 977 avec un échec d'authentification.

## P3-07 : structuration du lieu des transactions

Critère de fin : plus aucun manquant inexpliqué dans les colonnes de lieu. Voir `docs/07_lieu_transactions.md`.

In [ ]:
df_transaction = structurer_lieu_transactions(df_transaction)
check_location_structuration(df_transaction)

**Constats**

- Les colonnes de lieu n'ont plus aucune valeur manquante ni « UNKNOWN ».
- 1 563 700 transactions sont en ligne, 89 006 à l'étranger et 11 653 209 sur place aux États-Unis.
- 5 788 transactions sont signalées par `online_channel_mismatch`.

## P3-08 : règles pour les valeurs suspectes

Les règles viennent du rapport de diagnostic (`docs/P2-16_rapport_diagnostic.md`). Aucune ligne n'est supprimée et aucune valeur n'est modifiée : chaque cas est signalé par un indicateur 0/1.

Critère de fin : chaque règle appliquée et tracée. Voir `docs/08_valeurs_suspectes.md`.

In [ ]:
df_transaction = flag_suspicious_transactions(df_transaction)
df_users = flag_suspicious_users(df_users)
df_card = flag_suspicious_cards(df_card)

check_suspicious_flags(df_transaction, df_users, df_card)

**Constats**

- 660 049 montants négatifs et 10 639 montants nuls sont signalés.
- 15 lignes sont signalées par `is_repeated` : une par groupe répété sans erreur, comme prévu dans P2-14.
- 15 clients sont signalés par `suspicious_income`. C'est le même nombre que les revenus par habitant nuls : les 8 revenus annuels inférieurs à 1 000 en font tous partie.
- 31 cartes sont signalées par `zero_credit_limit`.
- Aucune ligne n'est supprimée et aucune valeur n'est modifiée.